# 03 - Data Quality Audit
## Prediksi Harga Mobil Bekas di Indonesia

Notebook ini mengidentifikasi masalah kualitas data dan mendokumentasikan temuan untuk menjadi dasar keputusan Data Cleaning.

**Penting:** notebook ini hanya melakukan audit. Jangan menghapus baris, mengimputasi nilai, mengubah satuan, atau memperbaiki data pada tahap ini.


## 1. Import Libraries


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")


## 2. Load Dataset

Baca dataset asli secara normal dan sebagai teks. Pembacaan teks membantu memeriksa representasi nilai pada CSV, khususnya format mileage.


In [2]:
candidate_paths = []

for folder in [Path.cwd(), *Path.cwd().parents]:
    candidate_paths.append(folder / "data" / "used_car.csv")
    candidate_paths.append(folder / "used_car.csv")

DATA_PATH = next((path.resolve() for path in candidate_paths if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "File used_car.csv tidak ditemukan. Letakkan file pada folder "
        "'data' di direktori utama proyek, lalu periksa Path.cwd()."
    )

df = pd.read_csv(DATA_PATH)
df_raw = pd.read_csv(DATA_PATH, dtype="string")

print("Lokasi dataset :", DATA_PATH)
print("Dimensi dataset:", df.shape)
print("Dimensi raw text:", df_raw.shape)

if df.shape != df_raw.shape:
    raise ValueError("Dimensi pembacaan normal dan raw text tidak sama. Periksa parsing CSV.")

display(df.head())


Lokasi dataset : C:\Users\Carbon\OneDrive\Documents\KULIAH UPN\SEMESTER 5\Prak DataScience\project\32_IF-C_bersifatopsional\Data\used_car.csv
Dimensi dataset: (609, 20)
Dimensi raw text: (609, 20)


,car name,brand,year,mileage (km),location,transmission,plate type,rear camera,sun roof,auto retract mirror,electric parking brake,map navigator,vehicle stability control,keyless push start,sports mode,360 camera view,power sliding door,auto cruise control,price (Rp),instalment (Rp|Monthly)
0,AYLA X 1.2,Daihatsu,2018,10.508,Jakarta Utara,Manual,even plate,0,0,0,0,0,0,0,0,0,0,0,101000000,2060000
1,AGYA TRD SPORTIVO 1.0,Toyota,2015,112.888,Bogor,Manual,even plate,0,0,0,0,0,0,0,0,0,0,0,82000000,1670000
2,X-TRAIL 2.5,Nissan,2015,118.429,Surabaya,Automatic,odd plate,0,0,0,0,0,0,1,0,1,0,0,169000000,3440000
3,YARIS S TRD 1.5,Toyota,2020,15.945,Tangerang Selatan,Automatic,odd plate,0,0,0,0,0,0,0,0,0,0,0,218000000,4440000
4,AGYA G 1.2,Toyota,2019,30.404,Jakarta Barat,Manual,odd plate,0,0,0,0,0,0,0,0,0,0,0,117000000,2380000


## 3. Pemeriksaan Struktur dan Tipe Data


In [3]:
structure_audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(dropna=True)
})

display(structure_audit)


,dtype,non_null,missing,missing_pct,n_unique
car name,object,609,0,0.000,172
brand,object,609,0,0.000,13
year,int64,609,0,0.000,14
mileage (km),float64,609,0,0.000,609
location,object,609,0,0.000,13
transmission,object,609,0,0.000,2
plate type,object,609,0,0.000,2
rear camera,int64,609,0,0.000,2
sun roof,int64,609,0,0.000,2
auto retract mirror,int64,609,0,0.000,2


## 4. Audit Missing Values


In [4]:
missing_audit = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)

display(missing_audit)

print("Total missing value:", int(df.isna().sum().sum()))


,missing_count,missing_percentage
car name,0,0.000
brand,0,0.000
year,0,0.000
mileage (km),0,0.000
location,0,0.000
transmission,0,0.000
plate type,0,0.000
rear camera,0,0.000
sun roof,0,0.000
auto retract mirror,0,0.000


Total missing value: 0


## 5. Audit Duplikasi

Pemeriksaan ini menghitung duplikasi baris secara persis. Duplikasi persis perlu ditinjau, tetapi belum otomatis dihapus.


In [5]:
duplicate_mask = df.duplicated(keep=False)
duplicate_rows = df.loc[duplicate_mask].copy()

print("Jumlah baris duplikat (semua kemunculan):", int(duplicate_mask.sum()))
print("Jumlah baris duplikat setelah kemunculan pertama:", int(df.duplicated().sum()))

if not duplicate_rows.empty:
    display(duplicate_rows.head(20))
else:
    print("Tidak ditemukan baris duplikat secara persis.")


Jumlah baris duplikat (semua kemunculan): 0
Jumlah baris duplikat setelah kemunculan pertama: 0
Tidak ditemukan baris duplikat secara persis.


## 6. Audit Tipe Data dan Kegagalan Konversi Numerik

Bandingkan kolom numerik yang diharapkan dengan hasil konversi. Nilai yang tidak dapat dikonversi perlu diperiksa, bukan langsung diperbaiki.


In [6]:
expected_numeric_columns = [
    "year",
    "mileage (km)",
    "price (Rp)",
    "instalment (Rp|Monthly)"
]

numeric_conversion_rows = []

for column in expected_numeric_columns:
    if column not in df_raw.columns:
        numeric_conversion_rows.append({
            "kolom": column,
            "status": "Kolom tidak ditemukan",
            "jumlah_gagal_konversi": np.nan,
            "contoh_nilai_gagal": None
        })
        continue

    raw_values = df_raw[column]
    converted = pd.to_numeric(raw_values, errors="coerce")

    failed_mask = raw_values.notna() & converted.isna()
    examples = raw_values.loc[failed_mask].drop_duplicates().head(10).tolist()

    numeric_conversion_rows.append({
        "kolom": column,
        "status": str(df[column].dtype) if column in df.columns else "Tidak tersedia",
        "jumlah_gagal_konversi": int(failed_mask.sum()),
        "contoh_nilai_gagal": examples
    })

numeric_conversion_audit = pd.DataFrame(numeric_conversion_rows)
display(numeric_conversion_audit)


,kolom,status,jumlah_gagal_konversi,contoh_nilai_gagal
0,year,int64,0,[]
1,mileage (km),float64,0,[]
2,price (Rp),int64,0,[]
3,instalment (Rp|Monthly),int64,0,[]


## 7. Pemeriksaan Format Mileage

Tampilkan nilai mileage dalam bentuk teks mentah dan bentuk hasil pembacaan Pandas. Tujuannya melihat apakah ada pemisah, simbol, atau format yang memerlukan verifikasi dari dokumentasi sumber.

**Jangan** mengasumsikan tanda titik sebagai pemisah ribuan atau desimal sebelum ada bukti dari sumber dataset.


In [7]:
if "mileage (km)" in df.columns:
    mileage_format_check = pd.DataFrame({
        "raw_csv_value": df_raw["mileage (km)"].head(30),
        "pandas_value": df["mileage (km)"].head(30),
        "pandas_dtype": str(df["mileage (km)"].dtype)
    })
    display(mileage_format_check)

    print("Contoh nilai mileage unik dari CSV:")
    display(df_raw["mileage (km)"].dropna().drop_duplicates().head(30))
else:
    print("Kolom mileage (km) tidak ditemukan.")


,raw_csv_value,pandas_value,pandas_dtype
0,10.508,10.508,float64
1,112.888,112.888,float64
2,118.429,118.429,float64
3,15.945,15.945,float64
4,30.404,30.404,float64
5,17.306,17.306,float64
6,12.211,12.211,float64
7,126.885,126.885,float64
8,33.464,33.464,float64
9,33.364,33.364,float64


Contoh nilai mileage unik dari CSV:


0      10.508
1     112.888
2     118.429
3      15.945
4      30.404
5      17.306
6      12.211
7     126.885
8      33.464
9      33.364
10     44.969
11     43.292
12     57.385
13     53.151
14     46.182
15     58.152
16    118.704
17     78.385
18     45.376
19     37.343
20     61.937
21     47.085
22        178
23     10.301
24     75.302
25     18.298
26     50.387
27     64.235
28     44.473
29     24.577
Name: mileage (km), dtype: string

## 8. Audit Nilai Kategorikal dan Whitespace

Periksa spasi di awal/akhir nilai kategori, string kosong, dan kategori yang menandakan informasi tidak diketahui. Pemeriksaan ini tidak mengubah nilai.


In [8]:
categorical_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()

categorical_audit_rows = []

unknown_tokens = {
    "unknown", "unk", "n/a", "na", "not available", "none", "null", "-"
}

for column in categorical_columns:
    values = df_raw[column].dropna().astype("string")
    stripped = values.str.strip()

    whitespace_count = int((values != stripped).sum())
    empty_after_strip = int(stripped.eq("").sum())
    unknown_count = int(stripped.str.lower().isin(unknown_tokens).sum())

    categorical_audit_rows.append({
        "kolom": column,
        "jumlah_kategori_unik": int(stripped.nunique()),
        "nilai_dengan_whitespace": whitespace_count,
        "string_kosong_setelah_strip": empty_after_strip,
        "unknown_or_placeholder": unknown_count
    })

categorical_audit = pd.DataFrame(categorical_audit_rows)
display(categorical_audit)


,kolom,jumlah_kategori_unik,nilai_dengan_whitespace,string_kosong_setelah_strip,unknown_or_placeholder
0,car name,172,0,0,0
1,brand,13,0,0,0
2,location,13,0,0,52
3,transmission,2,0,0,0
4,plate type,2,0,0,0


In [9]:
for column in categorical_columns:
    print(f"\n{column} — kategori dan frekuensi:")
    display(df[column].value_counts(dropna=False).head(30))



car name — kategori dan frekuensi:


car name
BRIO SATYA E 1.2              48
XPANDER ULTIMATE 1.5          38
CALYA G 1.2                   21
AYLA R 1.2                    19
RUSH S TRD SPORTIVO 1.5       18
BRIO RS 1.2                   17
AGYA G TRD 1.2                16
HR-V E 1.5                    15
AVANZA G 1.3                  14
MOBILIO E 1.5                 12
AYLA X 1.0                     9
YARIS S TRD 1.5                9
KIJANG INNOVA REBORN G 2.0     9
IGNIS GX 1.2                   8
XPANDER SPORT 1.5              7
KIJANG INNOVA G 2.0            7
TERIOS R 1.5                   6
CITY RS HATCHBACK 1.5          6
LIVINA VL 1.5                  6
ALMAZ LT LUX 1.5               6
BALENO HATCHBACK 1.4           6
XPANDER EXCEED 1.5             6
BR-V E 1.5                     6
SX4 S-CROSS 1.5                6
AGYA G 1.0                     5
AYLA R DLX 1.2                 5
MOBILIO RS 1.5                 5
HR-V SE 1.5                    5
HR-V E PLUS 1.5                5
IGNIS GL 1.2                   5
N


brand — kategori dan frekuensi:


brand
Toyota        171
Honda         145
Daihatsu       82
Mitsubishi     63
Suzuki         62
Nissan         33
Wuling         24
Mazda          19
Datsun          4
Hyundai         3
Chevrolet       1
Ford            1
BMW             1
Name: count, dtype: int64


location — kategori dan frekuensi:


location
Bogor                104
Tangerang Selatan     98
Bekasi                68
Jakarta Utara         54
Unknown               52
Jakarta Timur         50
Jakarta Barat         45
Bandung               45
Depok                 33
Jakarta Selatan       24
Jakarta Pusat         19
Surabaya               9
Malang                 8
Name: count, dtype: int64


transmission — kategori dan frekuensi:


transmission
Automatic    446
Manual       163
Name: count, dtype: int64


plate type — kategori dan frekuensi:


plate type
even plate    321
odd plate     288
Name: count, dtype: int64

## 9. Validasi Fitur Biner

Fitur berikut diharapkan menggunakan nilai 0 dan 1. Audit ini menghitung nilai selain 0/1 dan nilai yang gagal dikonversi.


In [10]:
binary_features = [
    "rear camera",
    "sun roof",
    "auto retract mirror",
    "electric parking brake",
    "map navigator",
    "vehicle stability control",
    "keyless push start",
    "sports mode",
    "360 camera view",
    "power sliding door",
    "auto cruise control"
]

binary_audit_rows = []

for column in binary_features:
    if column not in df_raw.columns:
        binary_audit_rows.append({
            "kolom": column,
            "jumlah_0": np.nan,
            "jumlah_1": np.nan,
            "jumlah_missing": np.nan,
            "jumlah_nilai_tidak_valid": np.nan,
            "nilai_tidak_valid": "Kolom tidak ditemukan"
        })
        continue

    raw_values = df_raw[column]
    numeric_values = pd.to_numeric(raw_values, errors="coerce")

    invalid_parse = raw_values.notna() & numeric_values.isna()
    invalid_binary = numeric_values.notna() & ~numeric_values.isin([0, 1])
    invalid_mask = invalid_parse | invalid_binary

    invalid_values = raw_values.loc[invalid_mask].drop_duplicates().head(10).tolist()

    binary_audit_rows.append({
        "kolom": column,
        "jumlah_0": int(numeric_values.eq(0).sum()),
        "jumlah_1": int(numeric_values.eq(1).sum()),
        "jumlah_missing": int(raw_values.isna().sum()),
        "jumlah_nilai_tidak_valid": int(invalid_mask.sum()),
        "nilai_tidak_valid": invalid_values
    })

binary_audit = pd.DataFrame(binary_audit_rows)
display(binary_audit)


,kolom,jumlah_0,jumlah_1,jumlah_missing,jumlah_nilai_tidak_valid,nilai_tidak_valid
0,rear camera,522,87,0,0,[]
1,sun roof,591,18,0,0,[]
2,auto retract mirror,323,286,0,0,[]
3,electric parking brake,590,19,0,0,[]
4,map navigator,603,6,0,0,[]
5,vehicle stability control,546,63,0,0,[]
6,keyless push start,555,54,0,0,[]
7,sports mode,524,85,0,0,[]
8,360 camera view,599,10,0,0,[]
9,power sliding door,600,9,0,0,[]


## 10. Validasi Rentang Nilai

Periksa nilai yang secara logis tidak valid, seperti harga nol/negatif, mileage negatif, tahun nol/negatif, atau tahun produksi di masa depan. Tahun masa depan dihitung berdasarkan tahun kalender saat notebook dijalankan.


In [11]:
CURRENT_YEAR = pd.Timestamp.today().year

range_audit_rows = []

def add_range_audit(column, series, condition_name, mask):
    valid_mask = series.notna()
    count = int((mask & valid_mask).sum())
    denominator = int(valid_mask.sum())
    percentage = (count / denominator * 100) if denominator else np.nan

    range_audit_rows.append({
        "kolom": column,
        "jenis_pemeriksaan": condition_name,
        "jumlah_bermasalah": count,
        "persentase_dari_nilai_valid": percentage
    })

for column in expected_numeric_columns:
    if column not in df.columns:
        continue

    values = pd.to_numeric(df[column], errors="coerce")

    if column in ["price (Rp)", "instalment (Rp|Monthly)"]:
        add_range_audit(column, values, "Nilai nol atau negatif", values <= 0)

    elif column == "mileage (km)":
        add_range_audit(column, values, "Mileage negatif", values < 0)

    elif column == "year":
        add_range_audit(column, values, "Tahun nol atau negatif", values <= 0)
        add_range_audit(column, values, "Tahun lebih besar dari tahun saat ini", values > CURRENT_YEAR)

range_audit = pd.DataFrame(range_audit_rows)
display(range_audit)
print("Tahun kalender saat audit:", CURRENT_YEAR)


,kolom,jenis_pemeriksaan,jumlah_bermasalah,persentase_dari_nilai_valid
0,year,Tahun nol atau negatif,0,0.000
1,year,Tahun lebih besar dari tahun saat ini,0,0.000
2,mileage (km),Mileage negatif,0,0.000
3,price (Rp),Nilai nol atau negatif,0,0.000
4,instalment (Rp|Monthly),Nilai nol atau negatif,0,0.000


Tahun kalender saat audit: 2026


## 11. Outlier Detection dengan IQR

IQR digunakan untuk menandai nilai yang berada di luar batas statistik. Flag outlier bukan bukti bahwa data salah dan bukan instruksi untuk menghapus observasi.

Audit dilakukan pada variabel numerik kontinu yang relevan. Interpretasikan tahun produksi secara hati-hati karena sifatnya diskrit dan rentangnya dipengaruhi cakupan dataset.


In [12]:
iqr_columns = [
    "price (Rp)",
    "mileage (km)",
    "year",
    "instalment (Rp|Monthly)"
]

iqr_audit_rows = []
iqr_flagged_indices = {}

for column in iqr_columns:
    if column not in df.columns:
        continue

    values = pd.to_numeric(df[column], errors="coerce").dropna()

    if values.empty:
        continue

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    full_values = pd.to_numeric(df[column], errors="coerce")
    mask = full_values.notna() & ((full_values < lower) | (full_values > upper))
    iqr_flagged_indices[column] = df.index[mask].tolist()

    count = int(mask.sum())
    valid_count = int(full_values.notna().sum())

    iqr_audit_rows.append({
        "kolom": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "batas_bawah": lower,
        "batas_atas": upper,
        "jumlah_flag_outlier": count,
        "persentase": (count / valid_count * 100) if valid_count else np.nan
    })

iqr_audit = pd.DataFrame(iqr_audit_rows)
display(iqr_audit)


,kolom,Q1,Q3,IQR,batas_bawah,batas_atas,jumlah_flag_outlier,persentase
0,price (Rp),"132,000,000.000","212,000,000.000","80,000,000.000","12,000,000.000","332,000,000.000",26,4.269
1,mileage (km),39.145,81.726,42.581,-24.726,145.597,7,1.149
2,year,"2,017.000","2,019.000",2.000,"2,014.000","2,022.000",14,2.299
3,instalment (Rp|Monthly),"2,690,000.000","4,320,000.000","1,630,000.000","245,000.000","6,765,000.000",26,4.269


In [13]:
# Tinjau contoh observasi yang ter-flag sebagai outlier harga.
if "price (Rp)" in iqr_flagged_indices:
    flagged_price_rows = df.loc[
        iqr_flagged_indices["price (Rp)"]
    ].sort_values("price (Rp)", ascending=False)

    print("Contoh observasi dengan harga ter-flag IQR:")
    display(flagged_price_rows.head(20))


Contoh observasi dengan harga ter-flag IQR:


,car name,brand,year,mileage (km),location,transmission,plate type,rear camera,sun roof,auto retract mirror,electric parking brake,map navigator,vehicle stability control,keyless push start,sports mode,360 camera view,power sliding door,auto cruise control,price (Rp),instalment (Rp|Monthly)
318,FORTUNER VRZ GR SPORT 2.8,Toyota,2022,15.713,Jakarta Timur,Automatic,odd plate,0,0,1,0,0,0,0,0,0,0,0,542000000,11040000
575,3 20I (CKD) 2.0,BMW,2019,47.554,Jakarta Selatan,Automatic,even plate,0,0,0,0,1,0,0,1,0,0,0,446000000,9090000
406,PAJERO SPORT DAKAR ULTIMATE 4X2 2.4,Mitsubishi,2019,43.295,Unknown,Automatic,odd plate,0,0,0,0,0,0,0,0,0,0,0,443000000,9030000
563,CX-5 ELITE 2.5,Mazda,2019,29.440,Bandung,Automatic,even plate,0,1,1,0,0,0,0,0,0,0,0,433000000,8820000
357,PAJERO SPORT DAKAR 4X2 2.4,Mitsubishi,2018,60.880,Jakarta Utara,Automatic,even plate,0,0,1,0,0,0,1,0,0,0,0,429000000,8740000
232,PAJERO SPORT DAKAR ULTIMATE 4X2 2.4,Mitsubishi,2018,73.296,Jakarta Timur,Automatic,even plate,1,0,0,0,0,0,1,0,0,0,0,428000000,8720000
70,CX-5 ELITE 2.5,Mazda,2019,45.271,Tangerang Selatan,Automatic,even plate,0,0,1,0,0,0,0,0,0,0,0,426000000,8680000
261,CX-5 GT 2.5,Mazda,2019,16.763,Jakarta Barat,Automatic,odd plate,0,0,1,0,0,0,0,0,0,0,0,424000000,8640000
255,SANTA FE CRDI 2.2,Hyundai,2018,52.497,Bekasi,Automatic,odd plate,0,1,0,1,0,0,0,0,0,0,0,417000000,8500000
559,PAJERO SPORT DAKAR 4X2 2.4,Mitsubishi,2016,28.906,Tangerang Selatan,Automatic,even plate,0,1,0,1,0,0,0,0,0,0,0,404000000,8230000


## 12. Audit Potensi Data Leakage: Cicilan dan Harga

Korelasi tinggi antara cicilan dan harga dapat menjadi indikasi bahwa cicilan dihitung berdasarkan harga. Namun, korelasi saja tidak membuktikan secara pasti bagaimana variabel tersebut dibuat.

Periksa korelasi dan hubungan rasio cicilan terhadap harga. Jangan masukkan cicilan sebagai fitur model sebelum asal-usul dan waktu ketersediaan variabel dipastikan.


In [14]:
if {"price (Rp)", "instalment (Rp|Monthly)"}.issubset(df.columns):
    leakage_check = df[["price (Rp)", "instalment (Rp|Monthly)"]].apply(
        pd.to_numeric, errors="coerce"
    ).dropna()

    if len(leakage_check) >= 2:
        pearson_value = leakage_check["price (Rp)"].corr(
            leakage_check["instalment (Rp|Monthly)"], method="pearson"
        )
        spearman_value = leakage_check["price (Rp)"].corr(
            leakage_check["instalment (Rp|Monthly)"], method="spearman"
        )

        print("Jumlah pasangan data valid:", len(leakage_check))
        print("Korelasi Pearson :", pearson_value)
        print("Korelasi Spearman:", spearman_value)

        positive_price = leakage_check["price (Rp)"] > 0
        ratio = (
            leakage_check.loc[positive_price, "instalment (Rp|Monthly)"]
            / leakage_check.loc[positive_price, "price (Rp)"]
        )

        print("\nRingkasan rasio cicilan terhadap harga:")
        display(ratio.describe().to_frame("instalment / price"))
    else:
        print("Pasangan data valid tidak cukup untuk menghitung korelasi.")
else:
    print("Kolom harga atau cicilan tidak ditemukan.")


Jumlah pasangan data valid: 609
Korelasi Pearson : 0.9999979190615441
Korelasi Spearman: 1.0

Ringkasan rasio cicilan terhadap harga:


,instalment / price
count,609.000
mean,0.020
std,0.000
min,0.020
25%,0.020
50%,0.020
75%,0.020
max,0.020


## 13. Tabel Ringkasan Temuan Data Quality Audit

Gabungkan temuan missing value, duplikasi, format, kategori placeholder, validasi rentang, dan outlier menjadi satu tabel. Tabel ini menjadi dasar penyusunan rencana Data Cleaning.


In [15]:
issue_rows = []

# Missing values per kolom
for column, count in df.isna().sum().items():
    if count > 0:
        issue_rows.append({
            "variabel": column,
            "jenis_permasalahan": "Missing value",
            "jumlah": int(count),
            "persentase": float(count / len(df) * 100),
            "dampak": "Informasi tidak tersedia pada sebagian observasi",
            "rekomendasi_awal": "Tinjau pola dan penyebab missing sebelum menentukan penanganan"
        })

# Exact duplicate rows
duplicate_count = int(df.duplicated().sum())
if duplicate_count > 0:
    issue_rows.append({
        "variabel": "Seluruh kolom",
        "jenis_permasalahan": "Duplikasi baris persis",
        "jumlah": duplicate_count,
        "persentase": duplicate_count / len(df) * 100,
        "dampak": "Observasi identik dapat memengaruhi distribusi dan evaluasi",
        "rekomendasi_awal": "Verifikasi apakah benar listing duplikat sebelum menghapus"
    })

# Failed numeric conversions
for _, row in numeric_conversion_audit.iterrows():
    count = row.get("jumlah_gagal_konversi")
    if pd.notna(count) and int(count) > 0:
        issue_rows.append({
            "variabel": row["kolom"],
            "jenis_permasalahan": "Gagal konversi numerik",
            "jumlah": int(count),
            "persentase": int(count) / len(df) * 100,
            "dampak": "Nilai tidak dapat digunakan sebagai numerik tanpa penanganan",
            "rekomendasi_awal": "Periksa format asli dan dokumentasi sumber"
        })

# Invalid binary values
for _, row in binary_audit.iterrows():
    count = row.get("jumlah_nilai_tidak_valid")
    if pd.notna(count) and int(count) > 0:
        issue_rows.append({
            "variabel": row["kolom"],
            "jenis_permasalahan": "Nilai fitur biner di luar 0/1",
            "jumlah": int(count),
            "persentase": int(count) / len(df) * 100,
            "dampak": "Definisi indikator biner tidak konsisten",
            "rekomendasi_awal": "Verifikasi nilai dan definisi fitur dari sumber"
        })

# Placeholder categories
for _, row in categorical_audit.iterrows():
    count = int(row["unknown_or_placeholder"])
    if count > 0:
        issue_rows.append({
            "variabel": row["kolom"],
            "jenis_permasalahan": "Kategori unknown/placeholder",
            "jumlah": count,
            "persentase": count / len(df) * 100,
            "dampak": "Informasi kategori tidak spesifik",
            "rekomendasi_awal": "Tentukan apakah kategori dipertahankan sebagai kategori tersendiri atau ditangani"
        })

# Range checks
for _, row in range_audit.iterrows():
    count = int(row["jumlah_bermasalah"])
    if count > 0:
        issue_rows.append({
            "variabel": row["kolom"],
            "jenis_permasalahan": row["jenis_pemeriksaan"],
            "jumlah": count,
            "persentase": float(row["persentase_dari_nilai_valid"]),
            "dampak": "Nilai mungkin tidak valid secara logis",
            "rekomendasi_awal": "Periksa observasi dan konteks sebelum mengubah data"
        })

# IQR flags
for _, row in iqr_audit.iterrows():
    count = int(row["jumlah_flag_outlier"])
    if count > 0:
        issue_rows.append({
            "variabel": row["kolom"],
            "jenis_permasalahan": "Flag outlier IQR",
            "jumlah": count,
            "persentase": float(row["persentase"]),
            "dampak": "Observasi berada di luar rentang IQR",
            "rekomendasi_awal": "Periksa validitas dan karakteristik observasi; jangan hapus otomatis"
        })

audit_summary = pd.DataFrame(
    issue_rows,
    columns=[
        "variabel",
        "jenis_permasalahan",
        "jumlah",
        "persentase",
        "dampak",
        "rekomendasi_awal"
    ]
)

if audit_summary.empty:
    print("Tidak ada temuan pada kategori pemeriksaan yang dirangkum.")
else:
    audit_summary = audit_summary.sort_values(
        ["variabel", "jenis_permasalahan"]
    ).reset_index(drop=True)
    display(audit_summary)


,variabel,jenis_permasalahan,jumlah,persentase,dampak,rekomendasi_awal
0,instalment (Rp|Monthly),Flag outlier IQR,26,4.269,Observasi berada di luar rentang IQR,Periksa validitas dan karakteristik observasi;...
1,location,Kategori unknown/placeholder,52,8.539,Informasi kategori tidak spesifik,Tentukan apakah kategori dipertahankan sebagai...
2,mileage (km),Flag outlier IQR,7,1.149,Observasi berada di luar rentang IQR,Periksa validitas dan karakteristik observasi;...
3,price (Rp),Flag outlier IQR,26,4.269,Observasi berada di luar rentang IQR,Periksa validitas dan karakteristik observasi;...
4,year,Flag outlier IQR,14,2.299,Observasi berada di luar rentang IQR,Periksa validitas dan karakteristik observasi;...
